In [0]:
%sql
USE CATALOG crp_arb_dev;

In [0]:
from pyspark.sql import functions as F

silver_df = spark.readStream.table("crp_arb_dev.silver.crypto_data_15m")

gold_15m_signal = silver_df\
    .withColumn("volatility_pct", 
        F.round(((F.col("high_price") - F.col("low_price")) / F.col("low_price")) * 100, 2)) \
    .withColumn("price_diff_pct", 
        F.round(((F.col("close") - F.col("open")) / F.col("open")) * 100, 2)) \
    .withColumn("market_signal", 
        F.when(F.col("price_diff_pct") > 0.5, "BULLISH")
         .when(F.col("price_diff_pct") < -0.5, "BEARISH")
         .otherwise("STABLE")) \
    .withColumn("risk_level", 
        F.when(F.col("volatility_pct") > 1.2, "HIGH RISK")
         .when(F.col("volatility_pct") > 0.6, "MEDIUM")
         .otherwise("LOW"))

container_path = "abfss://crp-arb-cnt@sadlsdev.dfs.core.windows.net"

query_15m = gold_15m_signal.writeStream \
    .format("delta") \
    .outputMode("append") \
    .option("checkpointLocation", f"{container_path}/_checkpoints/gold_15m") \
    .option("path", f"{container_path}/gold/crypto_data_15m") \
    .trigger(availableNow=True) \
    .toTable("gold.crypto_data_15m")
    
query_15m.awaitTermination()